# Heart Attack Prediction with Logistic Regression

**Machine Learning and Smart Systems — BA, SS 2026, Universities of Europe, Potsdam**

Objective: implement logistic regression to predict heart attack risk from patient data,
documenting each step as **What / Why / How / Result**.

Dataset: Heart Attack Prediction Dataset (Kaggle) — 8763 patients, binary target `Heart Attack Risk` (0/1).


## 0. Setup

**What:** Import libraries.
**Why:** pandas for data, scikit-learn for the preprocessing pipeline, model and metrics,
matplotlib/seaborn for the plots.
**How:** standard scientific Python stack (tools used in the course).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, accuracy_score,
                             precision_score, recall_score, f1_score)

RANDOM_STATE = 42
THRESHOLD = 0.45
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)


## 1. Dataset exploration

**What:** Load the CSV and inspect structure, missing values, class balance and correlations.
**Why:** We must understand the data before modelling — data types, categorical columns, whether
the target is balanced, and whether any feature relates to the target.
**How:** `read_csv`, `info`, `value_counts`, a class-balance plot and a numeric correlation plot.


In [ ]:
df = pd.read_csv("heart_attack_prediction_dataset.csv")
print("Shape:", df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
print("Total missing values:", df.isnull().sum().sum())


In [ ]:
balance = df["Heart Attack Risk"].value_counts(normalize=True).round(3)
print(balance)
print(f"\nBaseline accuracy (always predict majority class): {balance.max():.3f}")

plt.figure(figsize=(4,3.2))
sns.countplot(x="Heart Attack Risk", data=df)
plt.title("Target class balance")
plt.tight_layout(); plt.show()


**Result:** ~64% no-risk (0), ~36% risk (1). The dataset is **imbalanced**, so accuracy alone is
misleading — a model always predicting "0" already scores ~0.64. We rely on recall and F1 for the
risk class.


In [ ]:
# Numeric feature correlation with the target
tmp = df.copy()
tmp[["Systolic_BP","Diastolic_BP"]] = tmp["Blood Pressure"].str.split("/", expand=True).astype(float)
tmp = tmp.select_dtypes(include=["number"])
corr_target = tmp.corr()["Heart Attack Risk"].drop("Heart Attack Risk").sort_values()

plt.figure(figsize=(6,6))
sns.barplot(x=corr_target.values, y=corr_target.index)
plt.title("Numeric feature correlation with Heart Attack Risk")
plt.xlabel("Pearson correlation")
plt.tight_layout(); plt.show()
print(corr_target.round(4))


**Result:** All correlations are near zero (|r| < 0.02) — an early signal that the features carry
little linear information about the target.


## 2. Data preprocessing strategy

**What:** Separate features/target, split `Blood Pressure` into numbers, and build a
`ColumnTransformer` that standardizes numeric columns and one-hot-encodes categorical columns.
**Why:**
- `Patient ID` is a unique identifier — no predictive value, dropped.
- `Blood Pressure` is stored as text `"158/88"`; the model needs numbers → Systolic/Diastolic.
- Categorical columns (`Sex`, `Diet`, `Country`, `Continent`, `Hemisphere`) must be encoded;
  `OneHotEncoder(drop='first')` turns each category into 0/1 columns without redundancy.
- Logistic regression is scale-sensitive → numeric features are standardized.
- Using a `ColumnTransformer` keeps preprocessing in one object, which is clean and (inside a
  Pipeline) prevents data leakage.
**How:** pandas string split + `ColumnTransformer([StandardScaler, OneHotEncoder])`.


In [ ]:
X = df.drop(columns=["Patient ID", "Heart Attack Risk"])
y = df["Heart Attack Risk"]

# Feature engineering: split Blood Pressure text into two numeric columns
X[["Systolic_BP", "Diastolic_BP"]] = X["Blood Pressure"].str.split("/", expand=True).astype(float)
X = X.drop(columns=["Blood Pressure"])

num_cols = X.select_dtypes(include=["number"]).columns.tolist()
cat_cols = ["Sex", "Diet", "Country", "Continent", "Hemisphere"]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols),
])

print("Numeric columns:", len(num_cols))
print("Categorical columns:", cat_cols)


**Result:** A single `preprocessor` object that scales numeric features and one-hot-encodes the
categorical ones. After transformation the data has 48 features (numeric + one-hot dummies).


## 3. Train-test split (with stratification)

**What:** Split into 80% train / 20% test, keeping the class ratio.
**Why:** Stratification preserves the 64/36 balance so the test set is representative;
`random_state=42` makes it reproducible.
**How:** `train_test_split(..., stratify=y, random_state=42)`, then fit the preprocessor on **train
only** and transform both sets.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)

X_train_proc = preprocessor.fit_transform(X_train)   # fit on train only -> no leakage
X_test_proc  = preprocessor.transform(X_test)

print("Train:", X_train_proc.shape, " Test:", X_test_proc.shape)
print("Train target mean:", round(y_train.mean(),3), " Test target mean:", round(y_test.mean(),3))


**Result:** Both splits keep ~36% positive class. Features standardized/encoded consistently.


## 4. Logistic regression model setup & training

**What:** Configure and train logistic regression.
**Why:** Logistic regression models P(y=1|x) = 1 / (1 + e^-z), z = b0 + b1x1 + ... . It is
interpretable (coefficients = feature effects). `class_weight='balanced'` counteracts the 64/36
imbalance so the model does not ignore the risk class. `liblinear` handles this size well.
**How:** `LogisticRegression(...).fit(...)`.


In [ ]:
model = LogisticRegression(solver="liblinear", class_weight="balanced",
                           random_state=RANDOM_STATE)
model.fit(X_train_proc, y_train)
print("Model trained on", X_train_proc.shape[1], "features.")


## 5. Prediction generation & threshold

**What:** Generate probability scores and turn them into class predictions using a **0.45 threshold**.
**Why:** `predict_proba` gives the raw risk probability. Lowering the threshold from 0.5 to 0.45
makes the model flag "risk" more readily — in medicine catching at-risk patients (recall) is often
prioritized over precision.
**How:** `predict_proba` + a threshold comparison.


In [ ]:
probabilities = model.predict_proba(X_test_proc)[:, 1]
y_pred = (probabilities >= THRESHOLD).astype(int)
print("First 10 predicted probabilities:", np.round(probabilities[:10], 3))
print("Threshold used:", THRESHOLD)


## 6. Model evaluation

**What:** Compute accuracy, precision, recall, F1 and the confusion matrix.
**Why:** On imbalanced data these together tell the real story:
- **Accuracy** = overall correctness (inflated by the majority class).
- **Precision** = of predicted-risk, how many truly at risk.
- **Recall / sensitivity** = of truly-at-risk, how many we caught — key in medicine.
- **F1** = harmonic mean of precision and recall.
**How:** scikit-learn metric functions + a confusion-matrix heatmap.


In [ ]:
print("--- CONFUSION MATRIX ---")
print(confusion_matrix(y_test, y_pred))
print("\n--- PERFORMANCE METRICS ---")
print(classification_report(y_test, y_pred, target_names=["No Risk","Risk"], zero_division=0))
print(f"Baseline accuracy (majority class): {max(y_test.mean(), 1-y_test.mean()):.3f}")


In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(4.5,4))
ConfusionMatrixDisplay(cm, display_labels=["No Risk","Risk"]).plot(cmap="Blues", values_format="d", ax=plt.gca())
plt.title(f"Confusion Matrix (threshold = {THRESHOLD})")
plt.tight_layout(); plt.show()


**Result:** With the 0.45 threshold the model reaches **recall ~0.89** on the risk class (it
catches most at-risk patients) but **precision ~0.36** (many false alarms), and overall accuracy
~0.40 — below the 0.64 baseline. The model flags almost everyone as "risk". It maximizes recall at
the cost of precision, but still does not truly separate the classes.


## 6b. Cross-validation (was it luck?)

**What:** Re-evaluate with 5-fold stratified cross-validation instead of a single split.
**Why:** A single split could be lucky or unlucky. Cross-validation trains/tests on 5 partitions
and reports mean +/- std, showing whether performance is **stable**.
**How:** `cross_val_score` with `StratifiedKFold`, wrapping the preprocessor + model in a `Pipeline`
so preprocessing is refit inside each fold (no leakage). CV uses the model's default decision rule.


In [ ]:
cv_pipeline = Pipeline([
    ("pre", preprocessor),
    ("clf", LogisticRegression(solver="liblinear", class_weight="balanced", random_state=RANDOM_STATE)),
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for metric in ["accuracy", "precision", "recall", "f1"]:
    scores = cross_val_score(cv_pipeline, X, y, cv=cv, scoring=metric)
    print(f"{metric:10s} folds={np.round(scores,3)}  mean={scores.mean():.3f} +/- {scores.std():.3f}")


**Result:** Across all 5 folds accuracy stays ~0.50 with a tiny standard deviation (~0.01),
F1 ~0.41. The near-chance performance is **consistent across the whole dataset** — not an artefact
of one split. The model genuinely lacks predictive signal here.


## 7. Results interpretation — feature importance

**What:** Read the logistic regression coefficients as feature effects.
**Why:** After standardization, coefficient magnitude ~ influence; sign = direction (positive pushes
risk probability up). With 48 features we show the 15 largest by absolute value.
**How:** map coefficients to feature names via the `ColumnTransformer`.


In [ ]:
feature_names = num_cols + preprocessor.named_transformers_["cat"].get_feature_names_out(cat_cols).tolist()
coefs = pd.Series(model.coef_[0], index=feature_names)
top = coefs.reindex(coefs.abs().sort_values(ascending=False).index).head(15).sort_values()

plt.figure(figsize=(6.5,6))
sns.barplot(x=top.values, y=top.index)
plt.title("Top 15 logistic regression coefficients (|value|)")
plt.xlabel("Coefficient")
plt.tight_layout(); plt.show()
print(top.round(4))


**Result:** Even the largest coefficients are small; no feature dominates. Consistent with the
near-zero correlations from step 1.


## 8. Discussion & conclusion

**Feature importance & coefficients.** All coefficients are small; no single indicator strongly
drives the prediction, matching the near-zero feature-target correlations.

**Performance & medical significance.** With a 0.45 threshold and class balancing, the model reaches
high recall (~0.89) on the risk class but low precision (~0.36) and overall accuracy (~0.40, below
the 0.64 baseline). In medicine, high recall is desirable (few missed at-risk patients), but a
precision this low means most positive flags are false alarms — not clinically usable as is.

**Model limitations & why.** This Kaggle dataset is widely known to be (semi-)synthetic: the
features show essentially no relationship with the target, so any linear classifier performs near
chance regardless of threshold. This is a property of the data, not of logistic regression. Five-fold
cross-validation confirmed the result is stable (accuracy ~0.50 +/- 0.01), so the poor performance is
real and reproducible, not one unlucky split.

**Potential improvements.**
- Try non-linear models (decision tree / random forest) to check for non-linear signal.
- Feature engineering (interactions, binning age/BP).
- Threshold tuning to trade precision for recall depending on the clinical goal.
- Obtain a dataset with genuine predictive signal.

**Real-world application.** A validated version of this pipeline could support (not replace)
clinicians by flagging high-risk patients for follow-up — but only after validation on real clinical
data with adequate precision and recall.
